In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

## 1.初始化并调用模型

langchain提供了两种常见方法用来初始化模型：
- 使用init_chat_model方法，由langchain自动创建模型对象
- 使用不同模型对应的Model类，手动创建模型对象


### 1.1.init_chat_model
使用init_chat_model方法，langchain根据模型名称自动初始化与模型的连接，非常方便。

但需要注意的是：**如果使用我们必须在.env中配置好模型提供者的api_key**


In [2]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-v4-pro")

In [3]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 1.2.自定义模型

init_chat_model默认会根据模型名称自动确定模型的提供者、其base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [6]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen3.7-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [7]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.3.调整模型参数
除了修改模型提供者以外，init_chat_model方法允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [11]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen3.7-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5,
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.4.使用model类
其实init_chat_model方法底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [9]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [10]:
# 打印结果
print(type(model))

<class 'langchain_community.chat_models.tongyi.ChatTongyi'>


# 2.访问模型

LangChain提供了两个不同的方法来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 2.1.invoke
invoke方法是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [20]:
# 通过invoke方法访问模型，需要阻塞等待模型生成结果
response = model.invoke("月亮的首都是哪里？")

In [21]:
# 查看响应内容
print(response)

content='在现实中，**月亮（月球）没有首都**。因为月球是一个自然天体，不是国家、省份或任何人类政治实体，所以不存在“首都”的概念。目前人类也没有在月球上建立任何永久性的城市或定居点，只有各国发射的无人探测器和宇航员短暂登陆留下的痕迹。\n\n不过，如果你是从文化、神话或科幻的角度来问，会有以下几个有趣的答案：\n\n1. **神话传说角度**：在中国古代神话中，月球的“中心”或标志性建筑可以说是**广寒宫**（嫦娥、吴刚和玉兔居住的地方）。\n2. **科幻作品角度**：在许多描绘未来人类殖民月球的科幻作品中，月球会有自己的中心城市。例如，在罗伯特·海因莱因的经典科幻小说《月亮是一个严厉的女人》中，月球独立后的首府和最大城市是**月城（Luna City）**；在《机动战士高达》等动漫中，月球也有如“格拉纳达”、“冯·布朗”等大型圆顶城市。\n3. **流行语/网名**：有时候网友会开玩笑说“月亮的首都是地球”，因为月球是地球的卫星，一直被地球的引力“管辖”着。\n\n如果你是在问现实的科学常识，答案就是：**没有**。不过，随着人类航天技术的发展（如中国的“嫦娥工程”和国际月球科研站计划），也许在遥远的未来，月球上会建立起第一个人类常驻基地，但那时它大概率会被称为“科研站”或“基地”，而不是“首都”。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1106, 'prompt_tokens': 15, 'total_tokens': 1121, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 791, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-max', 'system_finger

## 2.2.流式访问

阻塞式调用需要等待较长时间才能看到AI返回的结果，而流式调用则可以实时看到AI返回的一个个词。

In [22]:
# 通过.stream方法实现流式访问
stream = model.stream("月亮的首都是哪里？")

# 流式访问会返回一个生成器对象，可以遍历生成器对象来实时获取AI的回复
print(type(stream))

<class 'generator'>


In [23]:
# 遍历stream结果，实时打印AI的回复
for chunk in stream:
    print(chunk.content, end="", flush=True)

月亮（月球）**没有首都**。

原因如下：

1. **月球不是国家**：“首都”是一个国家或政治实体的行政中心。月球是地球的天然卫星，不是一个国家，没有政府，也没有政治体系。
2. **没有永久居民**：目前月球上没有人类永久定居点，只有各国的无人探测器（如月球车、着陆器）以及曾经短暂停留过的宇航员。
3. **国际法限制**：根据1967年联合国制定的《外层空间条约》（Outer Space Treaty），月球及其自然资源是“全人类的共同财富”。**任何国家都不能对月球主张主权**，也不能将其据为己有。因此，没有任何国家可以在月球上划分领土或设立首都。

如果您是在问某部**科幻小说、电影、动漫或游戏**（比如《机动战士高达》、《苍穹浩瀚》等）中的设定，请告诉我具体的作品名称，我可以为您解答该作品虚构世界里的“月球首都”！

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent方法用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [24]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [26]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-v4-pro")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [27]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "月亮的首都是哪里？"}]
})

print(response)

{'messages': [HumanMessage(content='月亮的首都是哪里？', additional_kwargs={}, response_metadata={}, id='45a38ada-11de-4859-9b5f-803f3ac7cf71'), AIMessage(content='这是一个充满趣味的问题，既可以从科学常识回答，也可以从网络梗的角度来解读。\n\n**1. 科学现实层面：月亮没有首都**\n从地理和政治意义上讲，**月亮没有首都**。\n*   **首都**是一个国家或政治实体的中央行政机关所在地。\n*   月球是一颗天然卫星，不是主权国家。根据《外层空间条约》，月球不属于任何国家，没有政府、没有居民，因此也不会有首都。\n\n**2. 硬核天文+历史梗：静海基地**\n有些了解航天历史的人可能会开玩笑说，月球的首都是**“静海基地”**（Tranquility Base）。\n*   1969年，阿波罗11号在此着陆，阿姆斯特朗和奥尔德林成为了首批踏上月球的人类。\n*   他们把那片区域命名为“静海基地”，有点像一个“临时殖民地”的名字。不过，这只是一个历史登陆点，不是行政中心。\n\n**3. 中国神话梗：广寒宫**\n在中国文化语境下，这个问题的标准答案往往是**“广寒宫”**。\n*   嫦娥、玉兔和吴刚都住在那里，既然嫦娥是月球上最早的（神话）居民，那广寒宫自然就是月球的核心区域了。\n\n**你问的是不是那个网络流行梗？**\n如果你是在玩最近流行的一个梗，比如：\n——“月亮的首都是哪里？”\n——“（因为是‘月亮’，所以）是‘亮都’？” （冷幽默式谐音）\n\n总结：严谨地说没有首都是广寒宫，硬核地说是静海基地，玩梗的话可以自己创造一个。😄', additional_kwargs={'refusal': None, 'reasoning_content': '这个问题有点意思，首都通常是一个主权国家的政治中心，但月亮...连居民都没有，哪来的首都？\n\n我得先判断一下提问者是不是在玩梗。看这问法，可能是真的好奇，也可能是在玩"月球首都"这个网络梗。最保险的方式是：先严肃解释月亮不是主权实体，没有首都的概念；再顺便提一下这个梗，以防对方就是在玩梗。\n\n科学层面讲清楚：月球没有国家、没有政府、没有常住人

### 流式访问


In [36]:
# 通过stream方法实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "月亮的首都是哪里？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [37]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # 检查是否有实际内容
        print(token.content, end="", flush=True)  # 打印

这个问题其实是一个中文互联网上的经典脑筋急转弯梗，并不是一个地理或天文问题。

**答案通常是：月亮的首都是“玉兔”。**

**梗的出处和解释：**
1.  **首都的谐音**：“首都”这个词，如果拆开来看，“首”可以理解为“头、第一”，“都”理解为“都市、城市”。
2.  **玉兔捣药**：在中国神话传说中，月亮上住着玉兔，它每天都在捣药。
3.  **文字游戏**：把“玉兔”和“首都”连在一起，就变成了“玉兔首（手）都（捣）药”——因为玉兔是用手在捣药的，所以它“手都”了。**“首”通“手”，“都”通“都”**，合起来就是玉兔的手在捣药，简称“玉兔手都（首都）”。

所以，这是一个利用谐音和神话故事创造出来的幽默段子，月亮本身并没有真正的首都。